# MVPA L2 Results Notebook

This notebook is the manuscript-facing review layer for `mvpa_L2.md`. It follows the five planned aims: pattern identification, SAD-HC neural profile differences, clinical relevance, SCR convergence, oxytocin modulation, and sensitivity checks.

The notebook is intentionally tolerant of missing optional outputs. The primary FearNetwork and MemoryFearNetwork outputs are expected; whole-brain/Schaefer sensitivity results are displayed when present and skipped when absent.


## Visualization And Reporting Principles

The figure style follows common high-impact MVPA/RSA reporting conventions: show the decoding evidence first, then representational geometry, learning dynamics, clinical/SCR association matrices, and sensitivity checks. The layout is inspired by multivoxel pattern and representational similarity work such as Haxby et al. (Science, 2001; DOI: `10.1126/science.1063736`), Norman et al. (Trends in Cognitive Sciences, 2006; DOI: `10.1016/j.tics.2006.07.005`), and Kriegeskorte et al. (Frontiers in Systems Neuroscience, 2008; DOI: `10.3389/neuro.06.004.2008`).

Plotting choices:

- Use colorblind-safe Okabe-Ito colors for group and drug contrasts.
- Show estimates with 95% confidence intervals whenever model tables provide them.
- Keep heatmaps centered at zero for signed effects.
- Show individual subject points behind summary distributions when subject-level metrics are available.
- Save important panels to `stats/figures` as PNG and PDF for manuscript triage.


In [ ]:
from pathlib import Path
import os
import math
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 160)
pd.set_option('display.max_rows', 200)

OKABE = {
    'orange': '#E69F00',
    'sky': '#56B4E9',
    'green': '#009E73',
    'yellow': '#F0E442',
    'blue': '#0072B2',
    'vermillion': '#D55E00',
    'purple': '#CC79A7',
    'black': '#000000',
    'gray': '#7A7A7A',
}
GROUP_COLORS = {'SAD': OKABE['vermillion'], 'HC': OKABE['blue']}
DRUG_COLORS = {'Placebo': OKABE['gray'], 'Oxytocin': OKABE['green']}

sns.set_theme(style='ticks', context='paper', font='Arial', font_scale=1.05)
plt.rcParams.update({
    'figure.dpi': 140,
    'savefig.dpi': 300,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.linewidth': 0.8,
    'axes.labelsize': 9,
    'axes.titlesize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

root_env = os.environ.get('MVPA_L2_ROOT')
candidates = []
if root_env:
    candidates.append(Path(root_env))
candidates.extend([
    Path('/Users/xiaoqianxiao/projects/NARSAD/LSS/results/mvpa_l2'),
    Path('/gscratch/scrubbed/fanglab/xiaoqian/NARSAD/LSS/results/mvpa_l2'),
    Path('/output_dir/mvpa_l2'),
    Path('outputs/mvpa_l2'),
])
MVPA_ROOT = next((p for p in candidates if p.exists()), candidates[0])
RESULTS_ROOT = MVPA_ROOT.parent if MVPA_ROOT.name == 'mvpa_l2' else MVPA_ROOT
HARMONIZED_DIR = MVPA_ROOT / 'harmonized'
STATS_DIR = MVPA_ROOT / 'stats'
FIG_DIR = STATS_DIR / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f'MVPA_ROOT = {MVPA_ROOT}')
print(f'RESULTS_ROOT = {RESULTS_ROOT}')
print(f'STATS_DIR = {STATS_DIR}')
print(f'FIG_DIR = {FIG_DIR}')


In [ ]:
CORE_METRICS = [
    'Neural_Dist_Safety_Background',
    'Neural_ThreatLike_Safety',
    'Neural_SafetyLike_Safety',
    'Neural_Boundary_Separation',
    'Neural_Decision_Margin_CSS',
    'Neural_Safety_Trajectory_Slope',
    'Neural_Threat_Trajectory_Slope',
]

COMPANION_METRICS = [
    'Neural_Dist_Threat_Background',
    'Neural_Dist_Threat_Safety',
    'Neural_ThreatLike_Threat',
    'Neural_SafetyLike_Threat',
]

PRIMARY_CLINICAL = ['lsas_total', 'lsas_fear', 'lsas_avoid', 'dass_anxiety']
PRIMARY_SCR = ['SCR_SafetyMinusBackground', 'SCR_ThreatMinusSafety', 'SCR_Safety_Trajectory_Slope', 'SCR_Threat_Trajectory_Slope']
SCR_FLAGS = [
    'SCR_Physiological_Responder',
    'SCR_Simple_Acquisition_Differential_Learner',
    'SCR_Habituation_Adjusted_Learner',
    'SCR_Late_Phase_Sensitivity_Learner',
]

PATHS = {
    'subject_metrics': HARMONIZED_DIR / 'mvpa_l2_subject_metrics.csv',
    'scr_flags': HARMONIZED_DIR / 'scr_sensitivity_groups.csv',
    'aim1_scr_sensitivity': STATS_DIR / 'aim1_scr_sensitivity.csv',
    'aim2': STATS_DIR / 'aim2_group_difference.csv',
    'aim3': STATS_DIR / 'aim3_clinical_relevance.csv',
    'aim4': STATS_DIR / 'aim4_scr_convergence.csv',
    'aim5': STATS_DIR / 'aim5_oxytocin_modulation.csv',
    'primary_all': STATS_DIR / 'primary_models_all.csv',
    'sensitivity_all': STATS_DIR / 'sensitivity_models_all.csv',
    'summary_md': STATS_DIR / 'mvpa_l2_results_summary.md',
}

def read_csv_or_empty(path):
    path = Path(path)
    if not path.exists():
        print(f'Missing: {path}')
        return pd.DataFrame()
    df = pd.read_csv(path)
    print(f'Loaded {path.name}: {df.shape[0]} rows x {df.shape[1]} columns')
    return df

metrics_df = read_csv_or_empty(PATHS['subject_metrics'])
scr_flags_df = read_csv_or_empty(PATHS['scr_flags'])
aim1_scr_df = read_csv_or_empty(PATHS['aim1_scr_sensitivity'])
aim2_df = read_csv_or_empty(PATHS['aim2'])
aim3_df = read_csv_or_empty(PATHS['aim3'])
aim4_df = read_csv_or_empty(PATHS['aim4'])
aim5_df = read_csv_or_empty(PATHS['aim5'])
primary_all_df = read_csv_or_empty(PATHS['primary_all'])
sensitivity_df = read_csv_or_empty(PATHS['sensitivity_all'])


## Data Availability

This section checks whether the notebook has the files needed for each aim and verifies the available diagnostic groups, drug conditions, and feature spaces. Missing whole-brain/Schaefer outputs are reported as absent sensitivity checks, not as negative results.


In [ ]:
file_status = pd.DataFrame({
    'file': list(PATHS.keys()),
    'path': [str(p) for p in PATHS.values()],
    'exists': [Path(p).exists() for p in PATHS.values()],
})
display(file_status)

if not metrics_df.empty:
    id_cols = [c for c in ['FeatureSpace', 'Group', 'Drug'] if c in metrics_df.columns]
    if id_cols:
        display(metrics_df.groupby(id_cols, dropna=False).size().rename('n_rows').reset_index())
    if 'FeatureSpace' in metrics_df.columns:
        feature_spaces = sorted(metrics_df['FeatureSpace'].dropna().astype(str).unique())
    else:
        feature_spaces = []
else:
    feature_spaces = []

print(f'Available feature spaces: {feature_spaces}')
has_wholebrain = any(fs.lower() in {'schaefer', 'wholebrain', 'wholebrain_schaefer', 'wholebrainparcellation'} for fs in feature_spaces)
if not has_wholebrain:
    print('Whole-brain/Schaefer sensitivity output is absent. This is allowed by the current no-wholebrain workflow.')

if not scr_flags_df.empty:
    flag_cols = [c for c in SCR_FLAGS if c in scr_flags_df.columns]
    if flag_cols:
        display(scr_flags_df[flag_cols].sum().rename('n_subjects').reset_index().rename(columns={'index': 'SCR subgroup'}))


In [ ]:
def select_columns(df, preferred):
    return [c for c in preferred if c in df.columns]

def tidy_result_table(df, n=30):
    if df is None or df.empty:
        return pd.DataFrame()
    cols = select_columns(df, [
        'analysis', 'sensitivity', 'feature_space', 'FeatureSpace', 'Group', 'Drug', 'metric', 'metric_z',
        'clinical_score', 'clinical_score_z', 'scr_index', 'test', 'term', 'estimate', 'ci_low', 'ci_high',
        'p', 'p_value', 'q', 'n', 'n_sad_subjects', 'n_hc_subjects', 'n_clinical_outliers_removed',
        'n_metric_outliers_removed', 'r2', 'status', 'formula'
    ])
    out = df[cols].copy() if cols else df.copy()
    p_col = 'p' if 'p' in out.columns else 'p_value' if 'p_value' in out.columns else None
    if p_col:
        out['_p_sort'] = pd.to_numeric(out[p_col], errors='coerce')
        out = out.sort_values('_p_sort', na_position='last').drop(columns='_p_sort')
    return out.head(n)

def model_rows(df):
    if df is None or df.empty:
        return pd.DataFrame()
    out = df.copy()
    if 'status' in out.columns:
        out = out[out['status'].fillna('ok').eq('ok')]
    return out

def numeric_col(df, col):
    return pd.to_numeric(df[col], errors='coerce') if col in df.columns else pd.Series(np.nan, index=df.index)

def save_figure(fig, name):
    png = FIG_DIR / f'{name}.png'
    pdf = FIG_DIR / f'{name}.pdf'
    fig.savefig(png, bbox_inches='tight', dpi=300)
    fig.savefig(pdf, bbox_inches='tight')
    print(f'Saved {png}')
    print(f'Saved {pdf}')

def add_panel_label(ax, label):
    ax.text(-0.12, 1.06, label, transform=ax.transAxes, fontsize=11, fontweight='bold', va='top', ha='left')

def label_from_row(row, fields):
    values = []
    for field in fields:
        if field in row.index and pd.notna(row[field]):
            values.append(str(row[field]))
    return ' | '.join(values) if values else str(row.name)

def plot_forest(df, title, top_n=25, label_fields=None, filename=None, color_by_q=True):
    sub = model_rows(df)
    needed = {'estimate', 'ci_low', 'ci_high'}
    if sub.empty or not needed.issubset(sub.columns):
        print(f'No forest plot available for {title}.')
        return None
    sub = sub.copy()
    label_fields = label_fields or ['metric', 'clinical_score', 'scr_index', 'sensitivity', 'feature_space', 'test']
    sub['label'] = sub.apply(lambda r: label_from_row(r, label_fields), axis=1)
    p_col = 'p' if 'p' in sub.columns else 'p_value' if 'p_value' in sub.columns else None
    if p_col:
        sub['_p_sort'] = numeric_col(sub, p_col)
        sub = sub.sort_values('_p_sort', na_position='last').drop(columns='_p_sort')
    sub = sub.head(top_n)
    est = numeric_col(sub, 'estimate')
    lo = numeric_col(sub, 'ci_low')
    hi = numeric_col(sub, 'ci_high')
    valid = est.notna() & lo.notna() & hi.notna()
    sub = sub[valid]
    est = est[valid]
    lo = lo[valid]
    hi = hi[valid]
    if sub.empty:
        print(f'No finite estimates for {title}.')
        return None
    y = np.arange(len(sub))
    if color_by_q and 'q' in sub.columns:
        q = numeric_col(sub, 'q').fillna(np.inf)
        colors = np.where(q < 0.05, OKABE['vermillion'], OKABE['blue'])
    else:
        colors = OKABE['blue']
    fig, ax = plt.subplots(figsize=(7.2, max(2.8, 0.27 * len(sub))))
    ax.axvline(0, color='0.45', linewidth=0.8)
    ax.errorbar(est, y, xerr=[est - lo, hi - est], fmt='none', ecolor='0.55', elinewidth=0.9, capsize=2)
    ax.scatter(est, y, c=colors, s=28, zorder=3)
    ax.set_yticks(y)
    ax.set_yticklabels(sub['label'])
    ax.invert_yaxis()
    ax.set_xlabel('Model estimate with 95% CI')
    ax.set_title(title)
    sns.despine(ax=ax)
    fig.tight_layout()
    if filename:
        save_figure(fig, filename)
    plt.show()
    return fig

def signed_logp_heatmap(df, row_col, col_col, title, group_col=None, filename=None):
    sub = model_rows(df)
    if sub.empty or row_col not in sub.columns or col_col not in sub.columns or 'estimate' not in sub.columns:
        print(f'No heatmap available for {title}.')
        return None
    p_col = 'p' if 'p' in sub.columns else 'p_value' if 'p_value' in sub.columns else None
    if not p_col:
        print(f'No p-value column available for {title}.')
        return None
    sub = sub.copy()
    pvals = numeric_col(sub, p_col).clip(lower=1e-300)
    sub['signed_logp'] = np.sign(numeric_col(sub, 'estimate')) * -np.log10(pvals)
    if group_col and group_col in sub.columns:
        groups = [g for g in ['SAD', 'HC'] if g in set(sub[group_col].dropna())]
        if not groups:
            groups = sorted(sub[group_col].dropna().unique())
        fig, axes = plt.subplots(1, len(groups), figsize=(4.4 * len(groups), 4.4), squeeze=False)
        axes = axes.ravel()
        vmax = np.nanpercentile(np.abs(sub['signed_logp']), 95) if sub['signed_logp'].notna().any() else 1
        vmax = max(vmax, 1)
        for ax, group in zip(axes, groups):
            mat = sub[sub[group_col].eq(group)].pivot_table(index=row_col, columns=col_col, values='signed_logp', aggfunc='mean')
            sns.heatmap(mat, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax, linewidths=0.5, linecolor='white', cbar=ax is axes[-1], ax=ax)
            ax.set_title(str(group))
            ax.set_xlabel(col_col)
            ax.set_ylabel(row_col if ax is axes[0] else '')
    else:
        mat = sub.pivot_table(index=row_col, columns=col_col, values='signed_logp', aggfunc='mean')
        fig, ax = plt.subplots(figsize=(max(4.5, 0.55 * mat.shape[1]), max(3.5, 0.38 * mat.shape[0])))
        vmax = np.nanpercentile(np.abs(mat.values), 95) if np.isfinite(mat.values).any() else 1
        vmax = max(vmax, 1)
        sns.heatmap(mat, cmap='RdBu_r', center=0, vmin=-vmax, vmax=vmax, linewidths=0.5, linecolor='white', cbar_kws={'label': 'Signed -log10(p)'}, ax=ax)
        ax.set_xlabel(col_col)
        ax.set_ylabel(row_col)
    fig.suptitle(title, y=1.02)
    fig.tight_layout()
    if filename:
        save_figure(fig, filename)
    plt.show()
    return fig

def plot_subject_metric_grid(df, metrics, title, feature_space='FearNetwork', drug='Placebo', group_col='Group', filename=None, max_cols=3):
    if df is None or df.empty:
        print(f'No subject table available for {title}.')
        return None
    sub = df.copy()
    if 'FeatureSpace' in sub.columns:
        sub = sub[sub['FeatureSpace'].astype(str).eq(feature_space)]
    if drug and 'Drug' in sub.columns:
        sub = sub[sub['Drug'].astype(str).eq(drug)]
    metrics = [m for m in metrics if m in sub.columns]
    if not metrics or group_col not in sub.columns:
        print(f'No available metrics for {title}.')
        return None
    n_cols = min(max_cols, len(metrics))
    n_rows = math.ceil(len(metrics) / n_cols)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3.0 * n_cols, 2.65 * n_rows), squeeze=False)
    for ax, metric in zip(axes.ravel(), metrics):
        plot_df = sub[[group_col, metric]].copy()
        plot_df[metric] = pd.to_numeric(plot_df[metric], errors='coerce')
        plot_df = plot_df.dropna()
        order = [g for g in ['HC', 'SAD'] if g in set(plot_df[group_col])]
        palette = [GROUP_COLORS.get(g, '0.4') for g in order]
        sns.violinplot(data=plot_df, x=group_col, y=metric, order=order, palette=palette, inner=None, linewidth=0.8, cut=0, ax=ax)
        sns.stripplot(data=plot_df, x=group_col, y=metric, order=order, color='black', alpha=0.38, size=2.4, jitter=0.18, ax=ax)
        sns.pointplot(data=plot_df, x=group_col, y=metric, order=order, color='black', errorbar=('ci', 95), markers='_', linestyles='none', ax=ax)
        ax.set_title(metric.replace('Neural_', '').replace('_', ' '), fontsize=8)
        ax.set_xlabel('')
        ax.set_ylabel('Subject metric')
    for ax in axes.ravel()[len(metrics):]:
        ax.axis('off')
    fig.suptitle(title, y=1.01)
    fig.tight_layout()
    if filename:
        save_figure(fig, filename)
    plt.show()
    return fig


## Aim 1: Identify Multivoxel Patterns Of Vicarious Threat And Safety

Primary inference: within SAD-placebo and HC-placebo, `CSR` versus `CSS` decoding accuracy is tested against a subject-aware permutation null. Cross-group decoding and Haufe-map similarity are secondary mechanistic checks of whether the learned code generalizes across groups and whether the spatial pattern is similar.


In [ ]:
def payload_value(payload, *keys):
    if not isinstance(payload, dict):
        return None
    for key in keys:
        if key in payload:
            return payload[key]
    for value in payload.values():
        if isinstance(value, dict):
            found = payload_value(value, *keys)
            if found is not None:
                return found
    return None

def feature_space_dir(feature_space):
    candidates = [
        RESULTS_ROOT / feature_space,
        MVPA_ROOT / feature_space,
        Path('/output_dir') / feature_space,
        Path('/gscratch/scrubbed/fanglab/xiaoqian/NARSAD/LSS/results') / feature_space,
        Path('/Users/xiaoqianxiao/projects/NARSAD/LSS/results') / feature_space,
        Path('outputs/mvpa_l2') / feature_space,
    ]
    return next((p for p in candidates if p.exists()), candidates[0])

def load_joblib(path):
    try:
        import joblib
    except ImportError:
        print('joblib is not installed in this kernel.')
        return None
    path = Path(path)
    if not path.exists():
        return None
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        return joblib.load(path)

def load_aim1_results(feature_space='FearNetwork'):
    base = feature_space_dir(feature_space)
    paths = [base / 'checkpoints' / 'cell_06.joblib', base / 'cell_06.joblib']
    for path in paths:
        payload = load_joblib(path)
        if payload is None:
            continue
        result = payload_value(payload, 'results_11') or payload
        if isinstance(result, dict):
            return result, path
    return None, None

def aim1_result_rows(result, feature_space='FearNetwork'):
    if not isinstance(result, dict):
        return pd.DataFrame()
    rows = []
    rows.append({'feature_space': feature_space, 'test': 'SAD self-decoding', 'estimate': result.get('acc_sad_cv'), 'p_value': result.get('p_sad'), 'family': 'Primary'})
    rows.append({'feature_space': feature_space, 'test': 'HC self-decoding', 'estimate': result.get('acc_hc_cv'), 'p_value': result.get('p_hc'), 'family': 'Primary'})
    func = result.get('func_matrix')
    pmat = result.get('p_func_pvals')
    if func is not None and pmat is not None:
        func = np.asarray(func)
        pmat = np.asarray(pmat)
        if func.shape == (2, 2) and pmat.shape == (2, 2):
            rows.append({'feature_space': feature_space, 'test': 'SAD model tested on HC', 'estimate': func[0, 1], 'p_value': pmat[0, 1], 'family': 'Secondary'})
            rows.append({'feature_space': feature_space, 'test': 'HC model tested on SAD', 'estimate': func[1, 0], 'p_value': pmat[1, 0], 'family': 'Secondary'})
    rows.append({'feature_space': feature_space, 'test': 'SAD-HC Haufe cosine similarity', 'estimate': result.get('sim_spatial'), 'p_value': result.get('p_sim'), 'family': 'Secondary'})
    return pd.DataFrame(rows)

def plot_aim1(result, feature_space='FearNetwork', filename='aim1_decoding_summary'):
    if not isinstance(result, dict):
        print('Aim 1 checkpoint is not available yet.')
        return None
    rows = aim1_result_rows(result, feature_space)
    primary = rows[rows['test'].isin(['SAD self-decoding', 'HC self-decoding'])].copy()
    fig, axes = plt.subplots(1, 3, figsize=(10.5, 3.1), gridspec_kw={'width_ratios': [1.0, 1.15, 0.9]})
    add_panel_label(axes[0], 'A')
    if not primary.empty:
        colors = [GROUP_COLORS['SAD'] if 'SAD' in t else GROUP_COLORS['HC'] for t in primary['test']]
        axes[0].bar(primary['test'].str.replace(' self-decoding', ''), primary['estimate'], color=colors, edgecolor='black', linewidth=0.6)
        axes[0].axhline(0.5, color='0.35', linestyle='--', linewidth=0.9)
        for i, row in primary.reset_index(drop=True).iterrows():
            p = row.get('p_value')
            label = f'p={p:.3g}' if pd.notna(p) else 'p=NA'
            axes[0].text(i, row['estimate'] + 0.025, label, ha='center', va='bottom', fontsize=8)
    axes[0].set_ylim(0.35, 1.0)
    axes[0].set_ylabel('Forced-choice accuracy')
    axes[0].set_title('Within-group decoding')
    add_panel_label(axes[1], 'B')
    func = np.asarray(result.get('func_matrix')) if result.get('func_matrix') is not None else None
    if func is not None and func.shape == (2, 2):
        sns.heatmap(func, cmap='cividis', vmin=0.3, vmax=0.9, annot=True, fmt='.2f', cbar_kws={'label': 'Accuracy'}, xticklabels=['Test SAD', 'Test HC'], yticklabels=['Train SAD', 'Train HC'], ax=axes[1])
    else:
        axes[1].text(0.5, 0.5, 'Cross-decoding matrix missing', ha='center', va='center')
        axes[1].axis('off')
    axes[1].set_title('Cross-group generalization')
    add_panel_label(axes[2], 'C')
    sim = result.get('sim_spatial')
    p_sim = result.get('p_sim')
    if sim is not None:
        axes[2].bar(['Haufe similarity'], [sim], color=OKABE['purple'], edgecolor='black', linewidth=0.6)
        axes[2].axhline(0, color='0.35', linewidth=0.8)
        label = f'p={p_sim:.3g}' if p_sim is not None and pd.notna(p_sim) else 'p=NA'
        axes[2].text(0, sim + 0.04 * np.sign(sim if sim != 0 else 1), label, ha='center', va='bottom', fontsize=8)
        axes[2].set_ylim(min(-1, sim - 0.2), max(1, sim + 0.2))
    axes[2].set_ylabel('Cosine similarity')
    axes[2].set_title('Spatial pattern similarity')
    fig.suptitle(f'Aim 1: {feature_space} CSR vs CSS pattern identification', y=1.03)
    fig.tight_layout()
    save_figure(fig, filename)
    plt.show()
    return fig

aim1_result, aim1_path = load_aim1_results('FearNetwork')
print(f'Aim 1 checkpoint: {aim1_path}')
display(aim1_result_rows(aim1_result, 'FearNetwork'))
plot_aim1(aim1_result, 'FearNetwork')


## Aim 2: SAD-HC Neural Profile Difference

Primary inference: placebo SAD versus HC tests on the core neural metrics. These metrics capture geometry, classifier evidence, boundary confidence, safety updating, and threat maintenance. The direction is interpreted as SAD-HC difference, not automatically as impairment.


In [ ]:
display(tidy_result_table(aim2_df, n=80))
plot_forest(aim2_df, 'Aim 2: SAD-HC difference in core neural profile', top_n=30, label_fields=['metric', 'feature_space'], filename='aim2_group_difference_forest')
plot_subject_metric_grid(metrics_df, CORE_METRICS, 'Aim 2: core neural metrics by group under placebo', feature_space='FearNetwork', drug='Placebo', filename='aim2_core_metric_distributions')


## Aim 2 Learning Dynamics

Trajectory plots test whether `CSS` patterns move toward the `CS-` safety reference and whether `CSR` patterns persist toward reinstated `CSR` or `SHOCK` threat anchors. The shock/US panel is secondary and appears only when the cache contains the relevant target analysis.


In [ ]:
def find_trajectory_payload(feature_space='FearNetwork'):
    base = feature_space_dir(feature_space)
    paths = [
        base / 'intermediate' / 'stage14_trajectories.joblib',
        base / 'checkpoints' / 'cell_14.joblib',
        base / 'checkpoints' / 'cell_12_trajectories.joblib',
        base / 'cell_14.joblib',
    ]
    for path in paths:
        payload = load_joblib(path)
        if payload is None:
            continue
        results = payload_value(payload, 'results_13_2') or payload
        if isinstance(results, dict):
            return results, path
    return None, None

def summarize_trajectory(df):
    if df is None or df.empty:
        return pd.DataFrame()
    trial_col = next((c for c in ['trial', 'Trial', 'block', 'Block'] if c in df.columns), None)
    score_col = next((c for c in ['score', 'Score', 'similarity', 'Similarity'] if c in df.columns), None)
    group_col = next((c for c in ['Group', 'group'] if c in df.columns), None)
    if trial_col is None or score_col is None or group_col is None:
        print(f'Trajectory dataframe columns are not recognized: {list(df.columns)}')
        return pd.DataFrame()
    sub = df.copy()
    sub['trial'] = pd.to_numeric(sub[trial_col], errors='coerce')
    sub['score'] = pd.to_numeric(sub[score_col], errors='coerce')
    sub['Group'] = sub[group_col].astype(str)
    sub = sub.dropna(subset=['trial', 'score'])
    out = sub.groupby(['Group', 'trial'], dropna=False)['score'].agg(['mean', 'sem', 'count']).reset_index()
    out['sem'] = out['sem'].fillna(0)
    return out

def annotate_trial_p(ax, stats_df, y=1.65):
    if stats_df is None or stats_df.empty:
        return
    trial_col = next((c for c in ['trial', 'Trial', 'block', 'Block'] if c in stats_df.columns), None)
    p_col = next((c for c in ['p', 'p_value', 'pval', 'P', 'p_group'] if c in stats_df.columns), None)
    if trial_col is None or p_col is None:
        return
    temp = stats_df.copy()
    temp[trial_col] = pd.to_numeric(temp[trial_col], errors='coerce')
    temp[p_col] = pd.to_numeric(temp[p_col], errors='coerce')
    temp = temp.dropna(subset=[trial_col, p_col])
    for _, row in temp.iterrows():
        if row[p_col] < 0.05:
            ax.text(row[trial_col], y, f'p={row[p_col]:.2g}', ha='center', va='bottom', fontsize=7)
            ax.plot([row[trial_col] - 0.15, row[trial_col] + 0.15], [y - 0.03, y - 0.03], color='black', linewidth=0.8)

def plot_trajectory_axis(ax, df, stats_df, title, target_label, target_color, marker):
    summary = summarize_trajectory(df)
    add_panel_label(ax, title[0])
    if summary.empty:
        ax.text(0.5, 0.5, 'Trajectory data missing', ha='center', va='center', transform=ax.transAxes)
        ax.set_axis_off()
        return
    for group in ['SAD', 'HC']:
        sub = summary[summary['Group'].eq(group)]
        if sub.empty:
            continue
        color = GROUP_COLORS.get(group, '0.3')
        ax.plot(sub['trial'], sub['mean'], marker=marker, color=color, linewidth=1.5, markersize=4, label=group)
        ax.fill_between(sub['trial'], sub['mean'] - sub['sem'], sub['mean'] + sub['sem'], color=color, alpha=0.18, linewidth=0)
    ax.axhline(0, color='0.5', linestyle='--', linewidth=0.9, label='Start')
    ax.axhline(1, color=target_color, linewidth=1.0, label=target_label)
    annotate_trial_p(ax, stats_df)
    ax.set_title(title)
    ax.set_xlabel('Trial')
    ax.set_ylabel('Similarity score')
    ax.set_ylim(-1.0, 2.0)
    ax.legend(frameon=False, fontsize=7, loc='upper left')

def plot_learning_dynamics(feature_space='FearNetwork'):
    results, path = find_trajectory_payload(feature_space)
    print(f'Trajectory checkpoint: {path}')
    if not isinstance(results, dict):
        print('Trajectory payload is not available.')
        return None
    fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.3), sharey=True)
    plot_trajectory_axis(axes[0], results.get('data_safe'), results.get('stats_safe'), 'A. Safety trajectory\nTarget = CS-', 'Target CS-', OKABE['green'], 'o')
    plot_trajectory_axis(axes[1], results.get('data_threat'), results.get('stats_threat'), 'B. Threat maintenance\nTarget = reinstated CSR', 'Target CSR', OKABE['vermillion'], 's')
    shock_df = results.get('data_threat_shock')
    shock_stats = results.get('stats_threat_shock')
    plot_trajectory_axis(axes[2], shock_df, shock_stats, 'C. Threat acquisition\nTarget = Shock/US', 'Target shock/US', OKABE['purple'], '^')
    fig.suptitle(f'Aim 2 dynamics: {feature_space}', y=1.03)
    fig.tight_layout()
    save_figure(fig, 'aim2_learning_dynamics')
    plt.show()
    return fig

plot_learning_dynamics('FearNetwork')


## Aim 2 Companion Metrics

Companion metrics clarify specificity: whether any group difference is safety-specific, threat-specific, or reflects broader reorganization of the threat-safety neural space.


In [ ]:
companion_available = [m for m in COMPANION_METRICS if not metrics_df.empty and m in metrics_df.columns]
print(f'Available companion metrics: {companion_available}')
plot_subject_metric_grid(metrics_df, companion_available, 'Aim 2 companion metrics by group under placebo', feature_space='FearNetwork', drug='Placebo', filename='aim2_companion_metric_distributions')


## Aim 3: Clinical Relevance

Primary inference: within each diagnostic group, z-scored anxiety symptoms are modeled as a function of z-scored neural metrics plus available covariates. This section uses the exact post-Hyak Aim 3 table, including clinical and neural outlier counts.


In [ ]:
display(tidy_result_table(aim3_df, n=100))
signed_logp_heatmap(aim3_df, 'metric', 'clinical_score', 'Aim 3: signed clinical-neural association strength', group_col='Group', filename='aim3_clinical_heatmap')
plot_forest(aim3_df, 'Aim 3: strongest clinical-neural associations', top_n=35, label_fields=['Group', 'metric', 'clinical_score'], filename='aim3_clinical_forest')


## Aim 4: Physiological Convergence With SCR

Primary inference: SCR learning indices are modeled as a function of neural metrics, diagnosis, drug condition, and covariates. SCR responder/learner cohorts are sensitivity populations that test whether neural results are robust among participants with measurable peripheral learning.


In [ ]:
display(tidy_result_table(aim4_df, n=100))
signed_logp_heatmap(aim4_df, 'metric', 'scr_index', 'Aim 4: signed SCR-neural convergence strength', filename='aim4_scr_heatmap')
plot_forest(aim4_df, 'Aim 4: strongest SCR convergence tests', top_n=35, label_fields=['metric', 'scr_index', 'feature_space'], filename='aim4_scr_forest')

if not scr_flags_df.empty:
    flag_cols = [c for c in SCR_FLAGS if c in scr_flags_df.columns]
    if flag_cols:
        counts = scr_flags_df[flag_cols].sum().rename('n_subjects').reset_index().rename(columns={'index': 'SCR cohort'})
        display(counts)
        fig, ax = plt.subplots(figsize=(5.6, 2.8))
        sns.barplot(data=counts, y='SCR cohort', x='n_subjects', color=OKABE['green'], ax=ax)
        ax.set_xlabel('Number of subjects')
        ax.set_ylabel('')
        ax.set_title('SCR sensitivity cohort sizes')
        fig.tight_layout()
        save_figure(fig, 'aim4_scr_cohort_sizes')
        plt.show()


## Aim 5: Oxytocin Modulation

Primary inference: `neural_metric ~ Group * Drug + covariates`. The strongest mechanistic claim is a directional shift of SAD-oxytocin toward the HC-placebo reference on metrics that differ under placebo. This section emphasizes the interaction estimate and the metric distributions across the factorial design.


In [ ]:
display(tidy_result_table(aim5_df, n=100))
plot_forest(aim5_df, 'Aim 5: Group by drug interaction on core neural metrics', top_n=35, label_fields=['metric', 'feature_space'], filename='aim5_group_drug_forest')

if not metrics_df.empty and {'Group', 'Drug'}.issubset(metrics_df.columns):
    sub = metrics_df.copy()
    if 'FeatureSpace' in sub.columns:
        sub = sub[sub['FeatureSpace'].astype(str).eq('FearNetwork')]
    available = [m for m in CORE_METRICS if m in sub.columns]
    if available:
        n_cols = 2
        n_rows = math.ceil(len(available) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(6.4, 2.7 * n_rows), squeeze=False)
        sub['GroupDrug'] = sub['Group'].astype(str) + '-' + sub['Drug'].astype(str)
        order = ['HC-Placebo', 'HC-Oxytocin', 'SAD-Placebo', 'SAD-Oxytocin']
        palette = [OKABE['blue'], OKABE['sky'], OKABE['vermillion'], OKABE['orange']]
        for ax, metric in zip(axes.ravel(), available):
            plot_df = sub[['GroupDrug', metric]].copy()
            plot_df[metric] = pd.to_numeric(plot_df[metric], errors='coerce')
            plot_df = plot_df.dropna()
            sns.boxplot(data=plot_df, x='GroupDrug', y=metric, order=order, palette=palette, width=0.55, fliersize=0, linewidth=0.8, ax=ax)
            sns.stripplot(data=plot_df, x='GroupDrug', y=metric, order=order, color='black', alpha=0.32, size=2.1, jitter=0.18, ax=ax)
            ax.set_title(metric.replace('Neural_', '').replace('_', ' '), fontsize=8)
            ax.set_xlabel('')
            ax.set_ylabel('Subject metric')
            ax.tick_params(axis='x', rotation=35)
        for ax in axes.ravel()[len(available):]:
            ax.axis('off')
        fig.suptitle('Aim 5: core neural metrics across Group x Drug cells', y=1.01)
        fig.tight_layout()
        save_figure(fig, 'aim5_group_drug_metric_distributions')
        plt.show()


## Sensitivity Analyses

Sensitivity analyses are used to test robustness, not to replace unsupported primary findings. This section includes alternative feature spaces such as MemoryFearNetwork, SCR responder/learner cohorts, and Analysis 1 subgroup decoding when `aim1_scr_sensitivity.csv` is available. Whole-brain/Schaefer sensitivity is optional and may be absent.


In [ ]:
if sensitivity_df.empty:
    print('No Aim 2-5 sensitivity model table found yet.')
else:
    display(tidy_result_table(sensitivity_df, n=140))
    if 'sensitivity' in sensitivity_df.columns:
        display(sensitivity_df.groupby(['analysis', 'sensitivity'], dropna=False).size().rename('n_tests').reset_index())
    signed_logp_heatmap(sensitivity_df, 'metric', 'sensitivity', 'Sensitivity analyses: signed effect strength', filename='sensitivity_signed_heatmap')
    plot_forest(sensitivity_df, 'Sensitivity analyses: strongest effects', top_n=45, label_fields=['metric', 'scr_index', 'sensitivity', 'feature_space'], filename='sensitivity_forest')

if aim1_scr_df.empty:
    print('No Analysis 1 SCR subgroup decoding summary found yet. Run scripts/export_aim1_scr_sensitivity.py after subgroup Stage 6 jobs finish.')
else:
    display(tidy_result_table(aim1_scr_df, n=80))
    plot_forest(aim1_scr_df, 'Aim 1 SCR-subgroup decoding sensitivity', top_n=40, label_fields=['test', 'include_subjects_flag', 'feature_space'], filename='aim1_scr_sensitivity_forest', color_by_q=False)

if not sensitivity_df.empty and 'feature_space' in sensitivity_df.columns:
    sensitivity_feature_spaces = sorted(sensitivity_df['feature_space'].dropna().astype(str).unique())
    print(f'Sensitivity feature spaces in model table: {sensitivity_feature_spaces}')
    has_wholebrain_sensitivity = any(fs.lower() in {'schaefer', 'wholebrain', 'wholebrain_schaefer', 'wholebrainparcellation'} for fs in sensitivity_feature_spaces)
    if not has_wholebrain_sensitivity:
        print('Whole-brain/Schaefer sensitivity was not run or not available. This is expected for the current no-wholebrain workflow.')


## Integrated Primary Result Summary

This triage view combines the primary model tables. Lead manuscript interpretation with rows that survive the planned multiple-comparison families, then use nominal and sensitivity findings to guide mechanistic interpretation.


In [ ]:
if primary_all_df.empty:
    frames = [df for df in [aim2_df, aim3_df, aim4_df, aim5_df] if not df.empty]
    primary_view = pd.concat(frames, ignore_index=True, sort=False) if frames else pd.DataFrame()
else:
    primary_view = primary_all_df.copy()

if primary_view.empty:
    print('No primary model results available yet.')
else:
    display(tidy_result_table(primary_view, n=160))
    p_col = 'p' if 'p' in primary_view.columns else 'p_value' if 'p_value' in primary_view.columns else None
    if 'q' in primary_view.columns:
        q = numeric_col(primary_view, 'q')
        sig = primary_view[q < 0.05].copy()
        print(f'FDR-significant primary rows: {len(sig)}')
        display(tidy_result_table(sig, n=160))
    if p_col:
        p = numeric_col(primary_view, p_col)
        if 'q' in primary_view.columns:
            q = numeric_col(primary_view, 'q')
            nominal = primary_view[(p < 0.05) & ~(q < 0.05)].copy()
        else:
            nominal = primary_view[p < 0.05].copy()
        print(f'Nominal primary rows needing cautious interpretation: {len(nominal)}')
        display(tidy_result_table(nominal, n=120))


## Saved Markdown Summary

If `scripts/summarize_mvpa_l2_results.py` has been run, its compact Markdown report is displayed here for quick cross-checking.


In [ ]:
summary_path = PATHS['summary_md']
if summary_path.exists():
    display(Markdown(summary_path.read_text()))
else:
    print(f'Missing summary file: {summary_path}')


## Reporting Checklist

Before using these figures in a manuscript, confirm:

- Subject counts are reported for each `Group * Drug` cell and each SCR sensitivity cohort.
- Aim 1 primary decoding tests use subject-aware permutation inference.
- Aim 2 primary claims are based on the prespecified core neural metrics, with companion metrics used for specificity.
- Aim 3 uses z-scored symptom and neural variables with the documented outlier rule and covariates.
- Aim 4 separates full-sample SCR convergence from SCR responder/learner sensitivity cohorts.
- Aim 5 distinguishes HC-reference shifts, general drug effects, SAD-specific modulation, and null modulation.
- Optional whole-brain/Schaefer sensitivity is labeled absent or pending when not run.
- Figure files saved in `stats/figures` are checked at final print size and remain interpretable in grayscale.
